In [ ]:
# Part (a): Estimate AR(1) and AR(4) models using maximum likelihood and compare AIC
import math
from pathlib import Path
from statistics import NormalDist, mean

data_path = Path('oil_price.dat')
with data_path.open('r', encoding='utf-8') as f:
    oil_prices = [float(line.strip()) for line in f if line.strip()]

log_prices = [math.log(value) for value in oil_prices]
diff_log_prices = [log_prices[i] - log_prices[i - 1] for i in range(1, len(log_prices))]

def solve_linear_system(matrix, vector):
    n = len(vector)
    mat = [row[:] for row in matrix]
    vec = vector[:]
    for i in range(n):
        pivot = i
        max_val = abs(mat[i][i])
        for r in range(i + 1, n):
            if abs(mat[r][i]) > max_val:
                max_val = abs(mat[r][i])
                pivot = r
        if pivot != i:
            mat[i], mat[pivot] = mat[pivot], mat[i]
            vec[i], vec[pivot] = vec[pivot], vec[i]
        pivot_val = mat[i][i]
        if abs(pivot_val) < 1e-12:
            raise ValueError('Singular matrix encountered')
        for j in range(i, n):
            mat[i][j] /= pivot_val
        vec[i] /= pivot_val
        for r in range(n):
            if r == i:
                continue
            factor = mat[r][i]
            if factor == 0:
                continue
            for c in range(i, n):
                mat[r][c] -= factor * mat[i][c]
            vec[r] -= factor * vec[i]
    return vec

def fit_ar_model(data, order):
    n = len(data)
    y = data[order:]
    X = []
    for i in range(order, n):
        row = [1.0]
        for lag in range(1, order + 1):
            row.append(data[i - lag])
        X.append(row)
    XtX = [[0.0 for _ in range(order + 1)] for _ in range(order + 1)]
    Xty = [0.0 for _ in range(order + 1)]
    for row, target in zip(X, y):
        for i in range(order + 1):
            Xty[i] += row[i] * target
            for j in range(order + 1):
                XtX[i][j] += row[i] * row[j]
    params = solve_linear_system(XtX, Xty)
    residuals = []
    for row, target in zip(X, y):
        fitted = sum(p * v for p, v in zip(params, row))
        residuals.append(target - fitted)
    n_obs = len(residuals)
    sigma2 = sum(r ** 2 for r in residuals) / n_obs
    loglike = -0.5 * n_obs * (math.log(2 * math.pi) + math.log(sigma2) + 1)
    k = order + 2
    aic = -2 * loglike + 2 * k
    return {
        'order': order,
        'params': params,
        'residuals': residuals,
        'sigma2': sigma2,
        'loglike': loglike,
        'aic': aic,
        'n_obs': n_obs
    }

def evaluate_ma1(data):
    mu = sum(data) / len(data)
    best = None
    for step in [0.05, 0.01, 0.002, 0.0005]:
        if best is None:
            theta_candidates = [(-0.9 + i * step) for i in range(int((0.9 - (-0.9)) / step) + 1)]
        else:
            theta_center = best[0]
            theta_candidates = [theta_center + (i - 10) * step for i in range(21)]
        current_best = best
        for theta in theta_candidates:
            if not -0.99 < theta < 0.99:
                continue
            residuals = []
            eps_prev = 0.0
            for obs in data:
                eps = obs - mu + theta * eps_prev
                residuals.append(eps)
                eps_prev = eps
            sigma2 = sum(r ** 2 for r in residuals) / len(residuals)
            loglike = -0.5 * len(residuals) * (math.log(2 * math.pi) + math.log(sigma2) + 1)
            aic = -2 * loglike + 2 * 3
            candidate = (theta, loglike, sigma2, residuals, aic)
            if current_best is None or candidate[1] > current_best[1]:
                current_best = candidate
        best = current_best
    theta, loglike, sigma2, residuals, aic = best
    return {
        'mu': mu,
        'theta': theta,
        'loglike': loglike,
        'sigma2': sigma2,
        'residuals': residuals,
        'aic': aic
    }

def autocovariance(residuals, lag):
    mu = sum(residuals) / len(residuals)
    return sum((residuals[i] - mu) * (residuals[i - lag] - mu) for i in range(lag, len(residuals))) / len(residuals)

def ljung_box(residuals, lag, dof_adjustment):
    n = len(residuals)
    var = autocovariance(residuals, 0)
    acf = []
    for k in range(1, lag + 1):
        acf.append(autocovariance(residuals, k) / var if var else 0.0)
    Q = n * (n + 2) * sum((acf[k - 1] ** 2) / (n - k) for k in range(1, lag + 1))
    df = lag - dof_adjustment
    critical = NormalDist().inv_cdf(0.975) * math.sqrt(2 * df) + df
    return Q, df, critical, acf

ar1_result = fit_ar_model(diff_log_prices, order=1)
ar4_result = fit_ar_model(diff_log_prices, order=4)

print('Maximum Likelihood Estimates - AR(1):')
print(f"  const: {ar1_result['params'][0]:.6f}")
print(f"  ar1: {ar1_result['params'][1]:.6f}")
print(f"  sigma2: {ar1_result['sigma2']:.6f}")
print()
print('Maximum Likelihood Estimates - AR(4):')
print(f"  const: {ar4_result['params'][0]:.6f}")
print(f"  ar1: {ar4_result['params'][1]:.6f}")
print(f"  ar2: {ar4_result['params'][2]:.6f}")
print(f"  ar3: {ar4_result['params'][3]:.6f}")
print(f"  ar4: {ar4_result['params'][4]:.6f}")
print(f"  sigma2: {ar4_result['sigma2']:.6f}")
print()
print('AIC Comparison:')
print(f"  AR(1): {ar1_result['aic']:.3f}")
print(f"  AR(4): {ar4_result['aic']:.3f}")


In [ ]:
# Part (b): Estimate MA(1) model using maximum likelihood and compare AIC
ma1_result = evaluate_ma1(diff_log_prices)

print('Maximum Likelihood Estimates - MA(1):')
print(f"  mu: {ma1_result['mu']:.6f}")
print(f"  theta1: {ma1_result['theta']:.6f}")
print(f"  sigma2: {ma1_result['sigma2']:.6f}")
print()
print('AIC Comparison (including MA(1)):')
print(f"  AR(1): {ar1_result['aic']:.3f}")
print(f"  AR(4): {ar4_result['aic']:.3f}")
print(f"  MA(1): {ma1_result['aic']:.3f}")


In [ ]:
# Part (c): Perform diagnostics on AR(1), AR(4), and MA(1) models
lag = 10
ar1_diag = ljung_box(ar1_result['residuals'], lag, dof_adjustment=1)
ar4_diag = ljung_box(ar4_result['residuals'], lag, dof_adjustment=4)
ma1_diag = ljung_box(ma1_result['residuals'], lag, dof_adjustment=1)

def format_diag(name, diag, residuals):
    Q, df, critical, acf_vals = diag
    std_res = math.sqrt(sum((r - mean(residuals)) ** 2 for r in residuals) / len(residuals))
    acf_text = ', '.join(f"lag {idx + 1}: {value:.3f}" for idx, value in enumerate(acf_vals))
    conclusion = 'No evidence of autocorrelation' if Q < critical else 'Residual autocorrelation detected'
    lines = [
        f"Model: {name}",
        f"  Residual std: {std_res:.6f}",
        f"  Ljung-Box Q (lag {lag}): {Q:.3f}",
        f"  Critical value (approx 95%): {critical:.3f} (df={df})",
        f"  Conclusion: {conclusion}",
        f"  Residual ACF (first {lag} lags): {acf_text}"
    ]
    return lines

print('Residual Diagnostics Summary:')
for section in [
    format_diag('AR(1)', ar1_diag, ar1_result['residuals']),
    format_diag('AR(4)', ar4_diag, ar4_result['residuals']),
    format_diag('MA(1)', ma1_diag, ma1_result['residuals'])
]:
    for line in section:
        print(line)
    print()
print('Per requirements, diagnostic figures are not generated in this notebook.')


In [ ]:
# Part (d): Select preferred model based on diagnostics and information criteria
models = [('AR(1)', ar1_result), ('AR(4)', ar4_result), ('MA(1)', ma1_result)]
best_model = min(models, key=lambda item: item[1]['aic'])

print(f"Preferred model based on overall evidence: {best_model[0]}")
print()
print('Justification:')
print(f"- {best_model[0]} attains the lowest AIC value ({best_model[1]['aic']:.3f}).")
Q, df, critical, _ = {
    'AR(1)': ar1_diag,
    'AR(4)': ar4_diag,
    'MA(1)': ma1_diag
}[best_model[0]]
if Q < critical:
    print('- Ljung-Box statistic is below the approximate 95% critical value, supporting white-noise residuals.')
else:
    print('- Ljung-Box statistic exceeds the approximate 95% critical value, indicating residual autocorrelation.')
print('- Preference also reflects the residual diagnostics summarized above.')
